In [0]:
'''%run "/Users/l242514@lhr.nu.edu.pk/Online-retail-data-pipeline/notebooks/00_config/00_config"

In [0]:
'''%run "/Users/l242514@lhr.nu.edu.pk/Online-retail-data-pipeline/src/utilities/logger"

In [0]:
'''from pyspark.sql.functions import explode, element_at, split, current_timestamp, lit

init_logging_table(spark, TABLE_LOGS)

# Widgets for Parameterized Backfills & Runs
dbutils.widgets.text("source_path", "/Volumes/workspace/ebay_pipeline/raw/full_load/", "Source File/Folder Path")
dbutils.widgets.text("load_type", "FULL", "Load Type (FULL/INCREMENTAL)")
dbutils.widgets.text("batch_date", datetime.date.today().strftime("%Y-%m-%d"), "Batch Date (YYYY-MM-DD)")

source_path = dbutils.widgets.get("source_path")
load_type = dbutils.widgets.get("load_type").upper()
batch_date = dbutils.widgets.get("batch_date")

target_table = TABLE_BRONZE_FULL if load_type == "FULL" else TABLE_BRONZE_INC

def process_raw_to_bronze():
    # 3b. Read with Schema-on-Read + Rescue Column for Schema Drift Handling
    raw_df = (spark.read
              .schema(raw_response_schema)
              .option("multiLine", "true")
              .option("rescuedDataColumn", "_rescued_data")
              .json(source_path))

    # 3c. Explode and Enrich Metadata
    bronze_df = (raw_df
                 .select(explode("itemSummaries").alias("item"), "_rescued_data")
                 .select("item.*", "_rescued_data")
                 .withColumn("batch_id", lit(f"{load_type}_{batch_date}"))
                 .withColumn("ingestion_timestamp", current_timestamp())
                 .withColumn("ingestion_date", lit(batch_date))
                 .withColumn("load_type", lit(load_type))
                 .withColumn("source_api", lit("eBay Browse API v1"))
                 .withColumn("source_file", col("_metadata.file_path"))
                 .withColumn("source_category", element_at(split(col("_metadata.file_path"), "/"), -1))
                 .withColumn("load_timestamp", current_timestamp()))

    # Create table if it doesn't exist
    bronze_df.write.format("delta").option("mergeSchema", "true").mode("append").saveAsTable(target_table)
    
    # Clean staging write via Idempotent MERGE / Overwrite on batch_id
    # Clear existing batch to prevent duplicate inserts on rerun
    spark.sql(f"DELETE FROM {target_table} WHERE batch_id = '{load_type}_{batch_date}'")
    
    bronze_df.write.format("delta").option("mergeSchema", "true").mode("append").saveAsTable(target_table)
    
    count = bronze_df.count()
    return count, 0, 0  # inserted, updated, quarantined

# Execute wrapped in audit logging framework
run_step(spark, TABLE_LOGS, "Raw-to-Bronze", load_type, target_table, source_path, process_raw_to_bronze)

In [0]:
# Databricks notebook source
# COMMAND ----------
# MAGIC %run ../00_config/00_config

# COMMAND ----------
# 1. EXPLICIT PYSPARK IMPORTS (Prevents NameError: StructType is not defined)
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, 
    TimestampType, ArrayType, DoubleType
)
from datetime import datetime
import uuid

# 2. WIDGETS FOR PARAMETERIZED BACKFILLS (Phase 2 Rubric)
dbutils.widgets.text("load_type", "INCREMENTAL", "Load Type (FULL or INCREMENTAL)")
dbutils.widgets.text("batch_filter", "ebay_incremental_*.json", "Incremental File Pattern")

LOAD_TYPE = dbutils.widgets.get("load_type").upper()
BATCH_FILTER = dbutils.widgets.get("batch_filter")

# 3. DIRECT LOCAL REPO PATHS
REPO_BASE = "/Workspace/Users/ammaraa5688@gmail.com/Online-retail-data-pipeline"
RAW_FULL_DIR = f"{REPO_BASE}/data/raw/full_load"
RAW_INC_DIR = f"{REPO_BASE}/data/raw/incremental_load"

start_time = datetime.now()

# COMMAND ----------
if LOAD_TYPE == "FULL":
    # ENVELOPE SCHEMA FOR RAW FULL LOAD JSON FILES (No inferSchema)
    full_envelope_schema = StructType([
        StructField("total", LongType(), True),
        StructField("itemSummaries", ArrayType(
            StructType([
                StructField("itemId", StringType(), True),
                StructField("title", StringType(), True),
                StructField("categories", ArrayType(StructType([
                    StructField("categoryId", StringType(), True),
                    StructField("categoryName", StringType(), True)
                ])), True),
                StructField("price", StructType([
                    StructField("value", StringType(), True),
                    StructField("currency", StringType(), True)
                ]), True),
                StructField("seller", StructType([
                    StructField("username", StringType(), True)
                ]), True),
                StructField("condition", StringType(), True),
                StructField("itemCreationDate", StringType(), True)
            ])
        ), True)
    ])

    source_path = f"{RAW_FULL_DIR}/*.json"
    df_raw = spark.read.option("multiline", "true").schema(full_envelope_schema).json(source_path)

    # UNNEST AND INJECT AUDIT LINEAGE
    df_bronze = (
        df_raw
        .select(
            F.explode("itemSummaries").alias("item"),
            F.col("_metadata.file_path").alias("source_file")
        )
        .select(
            F.col("item.itemId").alias("itemId"),
            F.col("item.title").alias("title"),
            F.col("item.categories").alias("categories"),
            F.col("item.price").alias("price"),
            F.col("item.seller").alias("seller"),
            F.col("item.condition").alias("condition"),
            F.col("item.itemCreationDate").alias("itemCreationDate"),
            F.col("source_file"),
            F.current_timestamp().alias("load_timestamp"),
            F.lit("initial_full_load").alias("batch_id"),
            F.lit("full").alias("load_type"),
            F.to_date(F.current_timestamp()).alias("ingestion_date")
        )
    )

    count_rows = df_bronze.count()
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .partitionBy("ingestion_date")
        .saveAsTable("ebay_pipeline.bronze_ebay_listings_full")
    )

    end_time = datetime.now()
    log_entry = [(
        str(uuid.uuid4()), "Raw-to-Bronze", "bronze_ebay_listings_full",
        "full_load/*.json", start_time, end_time, "SUCCESS", count_rows, 0, None
    )]
    
    log_schema = StructType([
        StructField("log_id", StringType(), False),
        StructField("layer", StringType(), False),
        StructField("table_name", StringType(), False),
        StructField("parameter_processed", StringType(), True),
        StructField("execution_start_time", TimestampType(), False),
        StructField("execution_end_time", TimestampType(), True),
        StructField("status", StringType(), False),
        StructField("rows_inserted", LongType(), True),
        StructField("rows_updated", LongType(), True),
        StructField("error_message", StringType(), True)
    ])
    spark.createDataFrame(log_entry, schema=log_schema).write.format("delta").mode("append").saveAsTable("ebay_pipeline.pipeline_execution_logs")
    print(f"Full Load Complete: {count_rows} records written to ebay_pipeline.bronze_ebay_listings_full")

else:
    # STRICT SCHEMA FOR INCREMENTAL BATCHES
    incremental_schema = StructType([
        StructField("load_type", StringType(), True),
        StructField("batch_id", StringType(), True),
        StructField("ingestion_timestamp", StringType(), True),
        StructField("subcategory_partition", StringType(), True),
        StructField("raw_payload", StructType([
            StructField("itemId", StringType(), True),
            StructField("title", StringType(), True),
            StructField("categories", ArrayType(StructType([
                StructField("categoryId", StringType(), True),
                StructField("categoryName", StringType(), True)
            ])), True),
            StructField("price", StructType([
                StructField("value", StringType(), True),
                StructField("currency", StringType(), True)
            ]), True),
            StructField("seller", StringType(), True),
            StructField("condition", StringType(), True),
            StructField("itemCreationDate", StringType(), True)
        ]), True)
    ])

    source_path = f"{RAW_INC_DIR}/{BATCH_FILTER}"
    df_raw = spark.read.option("multiline", "true").schema(incremental_schema).json(source_path)

    df_bronze = (
        df_raw
        .withColumn("source_file", F.col("_metadata.file_path"))
        .withColumn("load_timestamp", F.current_timestamp())
        .withColumn("ingestion_date", F.to_date(F.current_timestamp()))
    )

    count_rows = df_bronze.count()
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .partitionBy("ingestion_date")
        .saveAsTable("ebay_pipeline.bronze_ebay_listings_incremental")
    )

    end_time = datetime.now()
    log_entry = [(
        str(uuid.uuid4()), "Raw-to-Bronze", "bronze_ebay_listings_incremental",
        BATCH_FILTER, start_time, end_time, "SUCCESS", count_rows, 0, None
    )]
    
    log_schema = StructType([
        StructField("log_id", StringType(), False),
        StructField("layer", StringType(), False),
        StructField("table_name", StringType(), False),
        StructField("parameter_processed", StringType(), True),
        StructField("execution_start_time", TimestampType(), False),
        StructField("execution_end_time", TimestampType(), True),
        StructField("status", StringType(), False),
        StructField("rows_inserted", LongType(), True),
        StructField("rows_updated", LongType(), True),
        StructField("error_message", StringType(), True)
    ])
    spark.createDataFrame(log_entry, schema=log_schema).write.format("delta").mode("append").saveAsTable("ebay_pipeline.pipeline_execution_logs")
    print(f"Incremental Bronze Ingestion Complete: {count_rows} records loaded into ebay_pipeline.bronze_ebay_listings_incremental")